In [2]:
from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.functions import col

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MMDS") \
    .master("local[*]") \
    .config("spark.driver.memory", "16g") \
    .config("spark.executor.memory", "16g") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/12/27 22:45:06 WARN Utils: Your hostname, MacBook-Pro-2.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.234 instead (on interface en0)
25/12/27 22:45:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/27 22:45:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Define schema and dataframes

In [4]:
schema_ratings = StructType([
    StructField("user_id", IntegerType(), False),
    StructField("item_id", IntegerType(), False),
    StructField("rating", IntegerType(), False),
    StructField("timestamp", IntegerType(), False)
])

schema_movies = StructType([
    StructField("item_id", IntegerType(), False),
    StructField("title", StringType(), False),
    StructField('genres', StringType(), False)
])

schema_users = StructType([
    StructField("user_id", IntegerType(), False),
    StructField("gender", StringType(), False),
    StructField('age', StringType(), False),
    StructField('occupation', IntegerType(), False),
    StructField('zip_code', StringType(), False)
])

In [5]:
train_ratings = spark.read.option("delimiter", "::").csv("./data/ratings_train.dat", schema=schema_ratings)
test_ratings = spark.read.option("delimiter", "::").csv("./data/ratings_test.dat", schema=schema_ratings)

In [6]:
movies = spark.read.option("delimiter", "::").csv("./data/movies.dat", schema=schema_movies)
movies = (
    movies
    .withColumn("year", substring("title", -5, 4).cast("int"))
    .withColumn("title", substring("title", 0, length("title") - 6))
    .withColumn("genres", split("genres", r"\|"))
)
movies.printSchema()

root
 |-- item_id: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- year: integer (nullable = true)



In [7]:
users = spark.read.option("delimiter", "::").csv("./data/users.dat", schema=schema_users)

users = (
    users
    .withColumn("gender", when(col("gender") == "F", 0).otherwise(1))
)

users.show()

+-------+------+---+----------+--------+
|user_id|gender|age|occupation|zip_code|
+-------+------+---+----------+--------+
|      1|     0|  1|        10|   48067|
|      2|     1| 56|        16|   70072|
|      3|     1| 25|        15|   55117|
|      4|     1| 45|         7|   02460|
|      5|     1| 25|        20|   55455|
|      6|     0| 50|         9|   55117|
|      7|     1| 35|         1|   06810|
|      8|     1| 25|        12|   11413|
|      9|     1| 25|        17|   61614|
|     10|     0| 35|         1|   95370|
|     11|     0| 25|         1|   04093|
|     12|     1| 25|        12|   32793|
|     13|     1| 45|         1|   93304|
|     14|     1| 35|         0|   60126|
|     15|     1| 25|         7|   22903|
|     16|     0| 35|         0|   20670|
|     17|     1| 50|         1|   95350|
|     18|     0| 18|         3|   95825|
|     19|     1|  1|        10|   48073|
|     20|     1| 25|        14|   55113|
+-------+------+---+----------+--------+
only showing top

## Movies profile

In [8]:
from pyspark.sql.functions import min, max

year_stats = movies.agg(
    min("year").alias("min_year"),
    max("year").alias("max_year")
).collect()[0]

min_year, max_year = year_stats["min_year"], year_stats["max_year"]

movies = movies.withColumn(
    "year_norm",
    (col("year") - min_year) / (max_year - min_year)
)

In [12]:
import re
from pyspark.sql import functions as F
from pyspark.ml.feature import (
    CountVectorizer,
    VectorAssembler,
    Normalizer
)

# ------------------------------------------------------
# Helper: SQL-safe column names
# ------------------------------------------------------
def safe_col(name: str) -> str:
    return re.sub(r"[^a-zA-Z0-9_]", "_", name)

# ------------------------------------------------------
# 1. movies is ALREADY LOADED
# Required columns:
#   item_id : int
#   genres  : array<string>
#   year_norm : double
# ------------------------------------------------------
# movies.printSchema()

# ------------------------------------------------------
# 2. Load movies_enriched (actors live here)
# ------------------------------------------------------
movies_enriched = (
    spark.read
    .option("header", True)
    .option("multiLine", True)
    .option("escape", '"')
    .csv("./data/movies_enriched.csv")
    .select(
        F.col("movieId").alias("item_id"),
        F.split(F.col("actors"), r"\|").alias("actors")
    )
)

# ------------------------------------------------------
# 3. Find top-30 actors globally
# ------------------------------------------------------
top_actors = (
    movies_enriched
    .select(F.explode("actors").alias("actor"))
    .groupBy("actor")
    .count()
    .orderBy(F.desc("count"))
    .limit(30)
)

top_actors_list = [r["actor"] for r in top_actors.collect()]

# ------------------------------------------------------
# 4. One-hot encode top actors (SQL-safe)
# ------------------------------------------------------
actor_cols = []

for actor in top_actors_list:
    col_name = f"actor_{safe_col(actor)}"

    movies_enriched = movies_enriched.withColumn(
        col_name,
        F.when(F.array_contains(F.col("actors"), actor), 1.0).otherwise(0.0)
    )

    actor_cols.append(col_name)

# ------------------------------------------------------
# 5. Select actor features only
# ------------------------------------------------------
actors_features = movies_enriched.select(
    "item_id",
    *actor_cols
)

# ------------------------------------------------------
# 6. Join actor features → movies
# ------------------------------------------------------
movies = (
    movies
    .join(actors_features, on="item_id", how="left")
)

# Fill missing actor flags with zeros
movies = movies.fillna(0.0, subset=actor_cols)

# ------------------------------------------------------
# 7. Genres CountVectorizer (binary)
# ------------------------------------------------------
cv = CountVectorizer(
    inputCol="genres",
    outputCol="tf",
    vocabSize=18,
    minDF=1,
    binary=True
)

cv_model = cv.fit(movies)
tf_df = cv_model.transform(movies)

# ------------------------------------------------------
# 8. Assemble features
# ------------------------------------------------------
assembler = VectorAssembler(
    inputCols=["tf", "year_norm"] + actor_cols,
    outputCol="features_raw"
)

movie_features = assembler.transform(tf_df)

# ------------------------------------------------------
# 9. Normalize
# ------------------------------------------------------
normalizer = Normalizer(
    inputCol="features_raw",
    outputCol="features_norm",
    p=2
)

movies_profiles = normalizer.transform(movie_features)

# ------------------------------------------------------
# 10. Final output
# ------------------------------------------------------
movies_profiles.select(
    "item_id",
    "features_norm"
).show(truncate=False)

+-------+------------------------------------------------------------------------------------------------------------------------+
|item_id|features_norm                                                                                                           |
+-------+------------------------------------------------------------------------------------------------------------------------+
|1      |(49,[1,8,14,18],[0.5076499507001783,0.5076499507001783,0.5076499507001783,0.4763135339902907])                          |
|2      |(49,[6,8,15,18,29],[0.45266233056115557,0.45266233056115557,0.45266233056115557,0.4247202113907138,0.45266233056115557])|
|3      |(49,[1,4,18],[0.5892194800958428,0.5892194800958428,0.5528479072504204])                                                |
|4      |(49,[0,1,18],[0.5892194800958428,0.5892194800958428,0.5528479072504204])                                                |
|5      |(49,[1,18],[0.7292563786835827,0.684240552838917])                        

In [13]:
print(tf_df.select("tf").first()["tf"].toArray())

[0. 1. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0.]


In [14]:
movies_profiles.select("features_norm").show()

+--------------------+
|       features_norm|
+--------------------+
|(49,[1,8,14,18],[...|
|(49,[6,8,15,18,29...|
|(49,[1,4,18],[0.5...|
|(49,[0,1,18],[0.5...|
|(49,[1,18],[0.729...|
|(49,[2,3,9,18,19]...|
|(49,[1,4,18,39],[...|
|(49,[6,8,18],[0.5...|
|(49,[2,18],[0.729...|
|(49,[2,3,6,18],[0...|
|(49,[0,1,4,18],[0...|
|(49,[1,5,18],[0.5...|
|(49,[8,14,18,43],...|
|(49,[0,18],[0.729...|
|(49,[2,4,6,18],[0...|
|(49,[0,3,18,19],[...|
|(49,[0,4,18],[0.5...|
|(49,[3,18,25],[0....|
|(49,[1,18],[0.729...|
|(49,[2,18],[0.729...|
+--------------------+
only showing top 20 rows


## Users profile

In [15]:
from pyspark.ml.stat import Summarizer
from pyspark.ml.feature import Normalizer

movie_vecs = movies_profiles.select(
    col("item_id"),
    col("features_norm")
)

user_movie_vectors = (
    train_ratings
    .join(broadcast(movie_vecs), on='item_id') ## broadcase here so that movie_vecs is moved to every executor, instead of shuffling ratings
    .select("user_id", "rating", "features_norm")
)

user_profiles = (
    user_movie_vectors
    .groupBy("user_id")
    .agg(
        Summarizer.mean(
            col("features_norm"),
            weightCol=col("rating")
        ).alias("user_features")
    )
)

normalizer = Normalizer(
    inputCol="user_features",
    outputCol="user_features_norm",
    p=2
)

user_profiles = normalizer.transform(user_profiles)

## LSH

In [29]:
from pyspark.ml.feature import BucketedRandomProjectionLSH

# Increase bucket length for better candidate generation
# Smaller bucket = more precise but may miss candidates
# Larger bucket = more candidates but slower
lsh = BucketedRandomProjectionLSH(
    inputCol="features_norm",
    outputCol="hashes",
    bucketLength=2.0,  # Increased from 1.0
    numHashTables=5    # Add multiple hash tables for better recall
)

lsh_model = lsh.fit(movies_profiles)

movies_lsh = movies_profiles.select(
    col("item_id"),
    col("features_norm")
).cache()

users_lsh = user_profiles.select(
    col("user_id"),
    col("user_features_norm").alias("features_norm")
).cache()


25/12/27 22:52:08 WARN CacheManager: Asked to cache already cached data.
25/12/27 22:52:08 WARN CacheManager: Asked to cache already cached data.


In [30]:
from pyspark import StorageLevel

# Use a much higher threshold to get more candidates
# We'll filter by top-K ranking anyway, so better to have more candidates
recommendations = lsh_model.approxSimilarityJoin(
    users_lsh, 
    movies_lsh, 
    threshold=1.5,  # Increased from 0.5 to capture more candidates
    distCol="distance"
).select(
    col("datasetA.user_id").alias("user_id"),
    col("datasetB.item_id").alias("item_id"),
    col("distance")
).persist(StorageLevel.MEMORY_AND_DISK)

recommendations.count()

23376118

In [31]:
already_rated = train_ratings.select("user_id", "item_id")

In [37]:
from pyspark.sql.window import Window

recommendations = recommendations.join(
    already_rated,
    on=["user_id", "item_id"],
    how="left_anti"
)

window = Window.partitionBy("user_id").orderBy(col("distance").asc())
top_k = 1000

ranked_recs = (
    recommendations
    .withColumn("rank", row_number().over(window))
    .filter(col("rank") <= top_k)
)

In [38]:
rated_universe = (
    test_ratings
    .select("user_id", "item_id", "rating")
)

recs_on_rated = (
    ranked_recs
    .join(rated_universe, on=["user_id", "item_id"], how="inner")
)

eval_df = (
    recs_on_rated
    .withColumn("relevant", (col("rating") >= 4).cast("int"))
)

user_metrics = (
    eval_df
    .groupBy("user_id")
    .agg(
        (sum("relevant") / lit(top_k)).alias("precision"),
        sum("relevant").alias("hits"),
        count("*").alias("rated_recommended")
    )
    .join(
        rated_universe
        .filter(col("rating") >= 4)
        .groupBy("user_id")
        .count()
        .withColumnRenamed("count", "total_relevant"),
        on="user_id",
        how="left"
    )
    .fillna(0)
    .withColumn(
        "recall",
        when(col("total_relevant") > 0,
             col("hits") / col("total_relevant"))
        .otherwise(lit(0))
    )
)

avg_metrics = user_metrics.agg(
    avg("precision").alias(f"avg_precision@{top_k}"),
    avg("recall").alias(f"avg_recall@{top_k}")
)

avg_metrics.show()

+--------------------+------------------+
|  avg_precision@1000|   avg_recall@1000|
+--------------------+------------------+
|0.005181054937032277|0.3725746870290931|
+--------------------+------------------+



## Debug: Check recommendation coverage

In [33]:
# Check how many recommendations per user
recs_per_user = (
    ranked_recs
    .groupBy("user_id")
    .count()
    .agg(
        avg("count").alias("avg_recs_per_user"),
        min("count").alias("min_recs"),
        max("count").alias("max_recs")
    )
)
print("Recommendations per user:")
recs_per_user.show()

# Check total users with recommendations
total_users_with_recs = ranked_recs.select("user_id").distinct().count()
total_test_users = test_ratings.select("user_id").distinct().count()
print(f"Users with recommendations: {total_users_with_recs} / {total_test_users}")

# Check overlap with test set
overlap = recs_on_rated.groupBy("user_id").count()
print("\nAverage items recommended that were also rated in test:")
overlap.agg(avg("count").alias("avg_overlap")).show()

Recommendations per user:


+-----------------+--------+--------+
|avg_recs_per_user|min_recs|max_recs|
+-----------------+--------+--------+
|           1000.0|    1000|    1000|
+-----------------+--------+--------+



Users with recommendations: 6040 / 6040

Average items recommended that were also rated in test:


+------------------+
|       avg_overlap|
+------------------+
|3.9815116911364874|
+------------------+



In [28]:
# Check distance distribution
print("Distance distribution in recommendations:")
recommendations.select("distance").summary("count", "min", "25%", "50%", "75%", "max", "mean").show()

# Check if there are candidates beyond threshold
lsh_all_candidates = lsh_model.approxSimilarityJoin(
    users_lsh, 
    movies_lsh, 
    threshold=2.0,  # Much higher threshold
    distCol="distance"
).select(
    col("datasetA.user_id").alias("user_id"),
    col("datasetB.item_id").alias("item_id"),
    col("distance")
)

print(f"\nWith threshold=2.0: {lsh_all_candidates.count()} total pairs")
print("Distance distribution with threshold=2.0:")
lsh_all_candidates.select("distance").summary("count", "min", "25%", "50%", "75%", "max", "mean").show()

Distance distribution in recommendations:


+-------+-------------------+
|summary|           distance|
+-------+-------------------+
|  count|             900768|
|    min|0.13625966604028716|
|    25%|0.34535906207354167|
|    50%| 0.4191238985133514|
|    75%|0.46562483151963857|
|    max|0.49999990125341404|
|   mean| 0.3995580863679778|
+-------+-------------------+




With threshold=2.0: 14305467 total pairs
Distance distribution with threshold=2.0:


+-------+-------------------+
|summary|           distance|
+-------+-------------------+
|  count|           14305467|
|    min|0.13625966604028716|
|    25%| 0.7073721164768523|
|    50%| 0.8724917543997738|
|    75%|  1.007852474243879|
|    max| 1.4066505895195673|
|   mean|  0.850819453314098|
+-------+-------------------+



## Ideas to improve recall

In [39]:
# 1. Check if relevant items are even in the candidate set (before top-k filtering)
all_recs_on_rated = (
    recommendations  # Before filtering to top-k
    .join(already_rated, on=["user_id", "item_id"], how="left_anti")
    .join(rated_universe, on=["user_id", "item_id"], how="inner")
    .withColumn("relevant", (col("rating") >= 4).cast("int"))
)

max_possible_recall = (
    all_recs_on_rated
    .groupBy("user_id")
    .agg(
        sum("relevant").alias("hits_in_all_candidates")
    )
    .join(
        rated_universe
        .filter(col("rating") >= 4)
        .groupBy("user_id")
        .count()
        .withColumnRenamed("count", "total_relevant"),
        on="user_id"
    )
    .withColumn("max_recall", col("hits_in_all_candidates") / col("total_relevant"))
    .agg(avg("max_recall").alias("avg_max_possible_recall"))
)

print("Maximum possible recall (if we used ALL candidates, not just top-1000):")
max_possible_recall.show()

# 2. Check average ranking of relevant items
relevant_item_ranks = (
    ranked_recs
    .join(rated_universe.filter(col("rating") >= 4), on=["user_id", "item_id"], how="inner")
    .select("rank")
)

print("\nRank distribution of relevant items that made it to top-1000:")
relevant_item_ranks.summary("count", "min", "25%", "50%", "75%", "max", "mean").show()

Maximum possible recall (if we used ALL candidates, not just top-1000):


+-----------------------+
|avg_max_possible_recall|
+-----------------------+
|     0.9972477217346981|
+-----------------------+


Rank distribution of relevant items that made it to top-1000:


+-------+------------------+
|summary|              rank|
+-------+------------------+
|  count|             28387|
|    min|                 1|
|    25%|               181|
|    50%|               463|
|    75%|               729|
|    max|              1000|
|   mean|465.98664881812095|
+-------+------------------+



### Solution 1: Add semantic embeddings from movie overviews

The most impactful improvement - add rich text embeddings

In [40]:
# Load movie overview embeddings
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.sql.types import ArrayType, FloatType

embeddings_df = (
    spark.read
    .option("header", True)
    .csv("./data/movies_overview_embeddings.csv")
)

# Convert embedding columns to array
emb_cols = [f"emb_{i}" for i in range(64)]

embeddings_df = (
    embeddings_df
    .select(
        col("movieId").cast("int").alias("item_id"),
        *[col(c).cast("float") for c in emb_cols]
    )
    .withColumn(
        "overview_emb_array",
        array(*emb_cols)
    )
)

# Convert array to dense vector
@udf(returnType=VectorUDT())
def array_to_vector(arr):
    return Vectors.dense(arr)

embeddings_df = embeddings_df.withColumn(
    "overview_emb",
    array_to_vector(col("overview_emb_array"))
).select("item_id", "overview_emb")

embeddings_df.show(5, truncate=False)

+-------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [43]:
# Rebuild movie profiles WITH embeddings
# Join embeddings to the existing movie features

movies_with_emb = movie_features.join(
    embeddings_df,
    on="item_id",
    how="left"
)

# Assemble ALL features including embeddings
# Handle nulls by skipping movies without embeddings
assembler_v2 = VectorAssembler(
    inputCols=["tf", "year_norm"] + actor_cols + ["overview_emb"],
    outputCol="features_raw_v2",
    handleInvalid="skip"  # Skip rows with null embeddings
)

movie_features_v2 = assembler_v2.transform(movies_with_emb)

# Normalize
normalizer_v2 = Normalizer(
    inputCol="features_raw_v2",
    outputCol="features_norm_v2",
    p=2
)

movies_profiles_v2 = normalizer_v2.transform(movie_features_v2)

print(f"Original feature dimension: ~{len(actor_cols) + 18 + 1}")  # genres + year + actors
print(f"New feature dimension: ~{len(actor_cols) + 18 + 1 + 64}")  # + embeddings

movies_profiles_v2.select("item_id", "features_norm_v2").show(5, truncate=False)

Original feature dimension: ~49
New feature dimension: ~113
+-------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [44]:
# Rebuild user profiles with new movie features
movie_vecs_v2 = movies_profiles_v2.select(
    col("item_id"),
    col("features_norm_v2")
)

user_movie_vectors_v2 = (
    train_ratings
    .join(broadcast(movie_vecs_v2), on='item_id')
    .select("user_id", "rating", "features_norm_v2")
)

user_profiles_v2 = (
    user_movie_vectors_v2
    .groupBy("user_id")
    .agg(
        Summarizer.mean(
            col("features_norm_v2"),
            weightCol=col("rating")
        ).alias("user_features_v2")
    )
)

normalizer_user_v2 = Normalizer(
    inputCol="user_features_v2",
    outputCol="user_features_norm_v2",
    p=2
)

user_profiles_v2 = normalizer_user_v2.transform(user_profiles_v2)

print("User profiles rebuilt with enriched features")
user_profiles_v2.show(5)

User profiles rebuilt with enriched features


+-------+--------------------+---------------------+
|user_id|    user_features_v2|user_features_norm_v2|
+-------+--------------------+---------------------+
|     12|[0.24702289152554...| [0.45854556310083...|
|     22|[0.10840433924538...| [0.18554359766183...|
|     26|[0.19075675391141...| [0.31867379664496...|
|     27|[0.13906334325125...| [0.28236482182726...|
|     28|[0.24118433999341...| [0.44451942566287...|
+-------+--------------------+---------------------+
only showing top 5 rows


In [45]:
# Retrain LSH with enriched features
lsh_v2 = BucketedRandomProjectionLSH(
    inputCol="features_norm_v2",
    outputCol="hashes",
    bucketLength=2.0,
    numHashTables=5
)

lsh_model_v2 = lsh_v2.fit(movies_profiles_v2)

movies_lsh_v2 = movies_profiles_v2.select(
    col("item_id"),
    col("features_norm_v2")
).cache()

users_lsh_v2 = user_profiles_v2.select(
    col("user_id"),
    col("user_features_norm_v2").alias("features_norm_v2")
).cache()

print("LSH model retrained with enriched features")

LSH model retrained with enriched features


In [46]:
# Generate recommendations with enriched features
recommendations_v2 = lsh_model_v2.approxSimilarityJoin(
    users_lsh_v2, 
    movies_lsh_v2, 
    threshold=1.5,
    distCol="distance"
).select(
    col("datasetA.user_id").alias("user_id"),
    col("datasetB.item_id").alias("item_id"),
    col("distance")
).persist(StorageLevel.MEMORY_AND_DISK)

print(f"Generated {recommendations_v2.count()} candidate pairs")

Generated 22892382 candidate pairs


In [47]:
# Filter and rank recommendations
recommendations_v2 = recommendations_v2.join(
    already_rated,
    on=["user_id", "item_id"],
    how="left_anti"
)

ranked_recs_v2 = (
    recommendations_v2
    .withColumn("rank", row_number().over(window))
    .filter(col("rank") <= top_k)
)

# Evaluate
recs_on_rated_v2 = (
    ranked_recs_v2
    .join(rated_universe, on=["user_id", "item_id"], how="inner")
)

eval_df_v2 = (
    recs_on_rated_v2
    .withColumn("relevant", (col("rating") >= 4).cast("int"))
)

user_metrics_v2 = (
    eval_df_v2
    .groupBy("user_id")
    .agg(
        (sum("relevant") / lit(top_k)).alias("precision"),
        sum("relevant").alias("hits"),
        count("*").alias("rated_recommended")
    )
    .join(
        rated_universe
        .filter(col("rating") >= 4)
        .groupBy("user_id")
        .count()
        .withColumnRenamed("count", "total_relevant"),
        on="user_id",
        how="left"
    )
    .fillna(0)
    .withColumn(
        "recall",
        when(col("total_relevant") > 0,
             col("hits") / col("total_relevant"))
        .otherwise(lit(0))
    )
)

avg_metrics_v2 = user_metrics_v2.agg(
    avg("precision").alias(f"avg_precision@{top_k}"),
    avg("recall").alias(f"avg_recall@{top_k}")
)

print("\n===== RESULTS WITH MOVIE OVERVIEW EMBEDDINGS =====")
avg_metrics_v2.show()


===== RESULTS WITH MOVIE OVERVIEW EMBEDDINGS =====


+--------------------+-------------------+
|  avg_precision@1000|    avg_recall@1000|
+--------------------+-------------------+
|0.006247306221758745|0.44321407612958375|
+--------------------+-------------------+



### Other improvements to try

1. **Use TF-IDF instead of binary for genres** - gives weight to rare genres
2. **Increase top actors from 30 to 50-100** - more actor features
3. **Use cosine similarity directly** instead of Euclidean distance
4. **Hybrid with ALS** - combine with collaborative filtering from 003_ALS.ipynb

### Summary of Results

| Configuration | Recall@1000 | Improvement |
|--------------|-------------|-------------|
| **Original (broken)** | ~5% | Baseline (incomplete coverage) |
| **V1: Fixed LSH** | 37.3% | +645% (fixed threshold & buckets) |
| **V2: + Embeddings** | 44.3% | +19% (semantic features) |
| **V3: + Quick Wins** | **45.0%** | **+1.6%** (TF-IDF + 100 actors + exp scoring) |

**Key insights:**
- Maximum possible recall: 99.7% (features can find almost all relevant items)
- Current bottleneck: **Ranking quality** (relevant items are scattered in top-1000)
- Combined improvements: **800% increase** from broken baseline to optimized system

**To reach 50-60% recall:**
1. **Hybrid with ALS** (biggest impact) - Combine content + collaborative filtering
2. **Re-ranking strategy** - Use cosine similarity directly on top candidates
3. **More metadata** - Add directors, keywords, production companies
4. **Better embeddings** - Use larger models or domain-specific embeddings

## Quick Wins Implementation

Let's implement the quick improvements to push recall higher

### 1. TF-IDF for genres (instead of binary)

In [48]:
from pyspark.ml.feature import IDF

# Use TF-IDF instead of binary CountVectorizer
cv_v3 = CountVectorizer(
    inputCol="genres",
    outputCol="genre_tf",
    vocabSize=18,
    minDF=1,
    binary=False  # Changed from True - now we get term frequencies
)

cv_model_v3 = cv_v3.fit(movies)
tf_df_v3 = cv_model_v3.transform(movies)

# Apply IDF to get TF-IDF
idf_v3 = IDF(
    inputCol="genre_tf",
    outputCol="genre_tfidf"
)

idf_model_v3 = idf_v3.fit(tf_df_v3)
tfidf_df_v3 = idf_model_v3.transform(tf_df_v3)

print("TF-IDF applied to genres - rare genres now have higher weights")

TF-IDF applied to genres - rare genres now have higher weights


### 2. Increase top actors from 30 to 100

In [49]:
# Extract top 100 actors (instead of 30)
top_actors_v3 = (
    movies_enriched
    .select(F.explode("actors").alias("actor"))
    .groupBy("actor")
    .count()
    .orderBy(F.desc("count"))
    .limit(100)  # Increased from 30
)

top_actors_list_v3 = [r["actor"] for r in top_actors_v3.collect()]

# One-hot encode top 100 actors
actor_cols_v3 = []
movies_enriched_v3 = movies_enriched  # Start fresh

for actor in top_actors_list_v3:
    col_name = f"actor_{safe_col(actor)}"
    
    movies_enriched_v3 = movies_enriched_v3.withColumn(
        col_name,
        F.when(F.array_contains(F.col("actors"), actor), 1.0).otherwise(0.0)
    )
    
    actor_cols_v3.append(col_name)

actors_features_v3 = movies_enriched_v3.select(
    "item_id",
    *actor_cols_v3
)

print(f"Using {len(actor_cols_v3)} actors (increased from 30)")

Using 100 actors (increased from 30)


### 3. Combine all improvements

In [51]:
# Select only needed columns from tfidf_df_v3 to avoid conflicts
tfidf_clean = tfidf_df_v3.select(
    "item_id", "genre_tfidf", "year_norm"
)

# Join improved actor features
movies_v3 = tfidf_clean.join(
    actors_features_v3,
    on="item_id",
    how="left"
).fillna(0.0, subset=actor_cols_v3)

# Join embeddings
movies_with_emb_v3 = movies_v3.join(
    embeddings_df,
    on="item_id",
    how="left"
)

# Assemble: TF-IDF genres + year + 100 actors + embeddings
assembler_v3 = VectorAssembler(
    inputCols=["genre_tfidf", "year_norm"] + actor_cols_v3 + ["overview_emb"],
    outputCol="features_raw_v3",
    handleInvalid="skip"
)

movie_features_v3 = assembler_v3.transform(movies_with_emb_v3)

# Normalize
normalizer_v3 = Normalizer(
    inputCol="features_raw_v3",
    outputCol="features_norm_v3",
    p=2
)

movies_profiles_v3 = normalizer_v3.transform(movie_features_v3)

print(f"Feature dimensions:")
print(f"  - Genres (TF-IDF): 18")
print(f"  - Year: 1")
print(f"  - Actors: {len(actor_cols_v3)}")
print(f"  - Overview embeddings: 64")
print(f"  - Total: ~{18 + 1 + len(actor_cols_v3) + 64}")

Feature dimensions:
  - Genres (TF-IDF): 18
  - Year: 1
  - Actors: 100
  - Overview embeddings: 64
  - Total: ~183


In [52]:
# Build user profiles with improved features
movie_vecs_v3 = movies_profiles_v3.select(
    col("item_id"),
    col("features_norm_v3")
)

user_movie_vectors_v3 = (
    train_ratings
    .join(broadcast(movie_vecs_v3), on='item_id')
    .select("user_id", "rating", "features_norm_v3")
)

user_profiles_v3 = (
    user_movie_vectors_v3
    .groupBy("user_id")
    .agg(
        Summarizer.mean(
            col("features_norm_v3"),
            weightCol=col("rating")
        ).alias("user_features_v3")
    )
)

normalizer_user_v3 = Normalizer(
    inputCol="user_features_v3",
    outputCol="user_features_norm_v3",
    p=2
)

user_profiles_v3 = normalizer_user_v3.transform(user_profiles_v3)

print("User profiles built with improved features")

User profiles built with improved features


In [53]:
# Train LSH with improved features
lsh_v3 = BucketedRandomProjectionLSH(
    inputCol="features_norm_v3",
    outputCol="hashes",
    bucketLength=2.0,
    numHashTables=5
)

lsh_model_v3 = lsh_v3.fit(movies_profiles_v3)

movies_lsh_v3 = movies_profiles_v3.select(
    col("item_id"),
    col("features_norm_v3")
).cache()

users_lsh_v3 = user_profiles_v3.select(
    col("user_id"),
    col("user_features_norm_v3").alias("features_norm_v3")
).cache()

print("LSH model trained")

LSH model trained


In [55]:
# Generate recommendations
recommendations_v3 = lsh_model_v3.approxSimilarityJoin(
    users_lsh_v3, 
    movies_lsh_v3, 
    threshold=1.5,
    distCol="distance"
).select(
    col("datasetA.user_id").alias("user_id"),
    col("datasetB.item_id").alias("item_id"),
    col("distance")
).persist(StorageLevel.MEMORY_AND_DISK)

print(f"Generated {recommendations_v3.count()} candidate pairs")

Generated 22931035 candidate pairs


In [56]:
# Filter and rank with IMPROVED SCORING
# Add exponential decay to distance weighting
recommendations_v3 = recommendations_v3.join(
    already_rated,
    on=["user_id", "item_id"],
    how="left_anti"
)

# Create score: lower distance = higher score, with exponential decay
# score = exp(-distance * 2)  # The multiplier controls decay rate
recommendations_v3 = recommendations_v3.withColumn(
    "score",
    exp(-col("distance") * 2.0)  # Exponential decay
)

window_v3 = Window.partitionBy("user_id").orderBy(col("score").desc())

ranked_recs_v3 = (
    recommendations_v3
    .withColumn("rank", row_number().over(window_v3))
    .filter(col("rank") <= top_k)
)

print("Recommendations ranked with exponential distance weighting")

Recommendations ranked with exponential distance weighting


In [57]:
# Evaluate with all quick wins
recs_on_rated_v3 = (
    ranked_recs_v3
    .join(rated_universe, on=["user_id", "item_id"], how="inner")
)

eval_df_v3 = (
    recs_on_rated_v3
    .withColumn("relevant", (col("rating") >= 4).cast("int"))
)

user_metrics_v3 = (
    eval_df_v3
    .groupBy("user_id")
    .agg(
        (sum("relevant") / lit(top_k)).alias("precision"),
        sum("relevant").alias("hits"),
        count("*").alias("rated_recommended")
    )
    .join(
        rated_universe
        .filter(col("rating") >= 4)
        .groupBy("user_id")
        .count()
        .withColumnRenamed("count", "total_relevant"),
        on="user_id",
        how="left"
    )
    .fillna(0)
    .withColumn(
        "recall",
        when(col("total_relevant") > 0,
             col("hits") / col("total_relevant"))
        .otherwise(lit(0))
    )
)

avg_metrics_v3 = user_metrics_v3.agg(
    avg("precision").alias(f"avg_precision@{top_k}"),
    avg("recall").alias(f"avg_recall@{top_k}")
)

print("\n===== RESULTS WITH ALL QUICK WINS =====")
print("(TF-IDF genres + 100 actors + embeddings + exponential scoring)")
avg_metrics_v3.show()


===== RESULTS WITH ALL QUICK WINS =====
(TF-IDF genres + 100 actors + embeddings + exponential scoring)


+--------------------+-----------------+
|  avg_precision@1000|  avg_recall@1000|
+--------------------+-----------------+
|0.006227823977823...|0.449817168955041|
+--------------------+-----------------+



## Hybrid: Content-Based (V3) + Collaborative Filtering (ALS)

Combine the best of both worlds - content understanding with user behavior patterns

In [58]:
# Train ALS model on training data
from pyspark.ml.recommendation import ALS

print("Training ALS model...")
als = ALS(
    maxIter=20, 
    regParam=0.05, 
    userCol="user_id", 
    itemCol="item_id", 
    ratingCol="rating",
    coldStartStrategy="drop",
    rank=15,
    nonnegative=True  # Ensures non-negative predictions
)

als_model = als.fit(train_ratings)
print("ALS model trained")

Training ALS model...


ALS model trained


In [59]:
# Generate ALS recommendations for all users
print("Generating ALS recommendations...")

# Get all unique users from test set
test_users = test_ratings.select("user_id").distinct()

# Generate top-K recommendations from ALS
als_recs = als_model.recommendForUserSubset(test_users, top_k)

# Flatten ALS recommendations
als_recs_flat = (
    als_recs
    .withColumn("rec", explode("recommendations"))
    .select(
        col("user_id"),
        col("rec.item_id").alias("item_id"),
        col("rec.rating").alias("als_score")
    )
)

# Filter out already rated items
als_recs_flat = als_recs_flat.join(
    already_rated,
    on=["user_id", "item_id"],
    how="left_anti"
)

print(f"ALS generated {als_recs_flat.count()} recommendations")

Generating ALS recommendations...


ALS generated 5590489 recommendations


In [60]:
# Normalize scores for both systems to [0, 1] range
from pyspark.sql import Window as W

# Normalize content-based scores (V3)
content_window = W.partitionBy("user_id")

content_normalized = (
    ranked_recs_v3
    .withColumn("min_score", min("score").over(content_window))
    .withColumn("max_score", max("score").over(content_window))
    .withColumn(
        "content_score_norm",
        when(col("max_score") > col("min_score"),
             (col("score") - col("min_score")) / (col("max_score") - col("min_score"))
        ).otherwise(0.5)
    )
    .select("user_id", "item_id", "content_score_norm", "rank")
)

# Normalize ALS scores
als_window = W.partitionBy("user_id")

als_normalized = (
    als_recs_flat
    .withColumn("min_score", min("als_score").over(als_window))
    .withColumn("max_score", max("als_score").over(als_window))
    .withColumn(
        "als_score_norm",
        when(col("max_score") > col("min_score"),
             (col("als_score") - col("min_score")) / (col("max_score") - col("min_score"))
        ).otherwise(0.5)
    )
    .select("user_id", "item_id", "als_score_norm")
)

print("Scores normalized")

Scores normalized


In [ ]:
# Combine both recommendation systems with weighted fusion
# Weight: 0.6 for ALS (collaborative), 0.4 for content-based
alpha = 0.5 # Weight for ALS

hybrid_recs = (
    content_normalized
    .join(als_normalized, on=["user_id", "item_id"], how="full_outer")
    .fillna(0.0, subset=["content_score_norm", "als_score_norm"])
    .withColumn(
        "hybrid_score",
        alpha * col("als_score_norm") + (1 - alpha) * col("content_score_norm")
    )
)

# Rank by hybrid score
hybrid_window = W.partitionBy("user_id").orderBy(col("hybrid_score").desc())

ranked_hybrid = (
    hybrid_recs
    .withColumn("rank", row_number().over(hybrid_window))
    .filter(col("rank") <= top_k)
)

print(f"Hybrid recommendations created with alpha={alpha} (ALS weight)")
print(f"Total hybrid recommendations: {ranked_hybrid.count()}")

Hybrid recommendations created with alpha=0.5 (ALS weight)


In [ ]:
# Evaluate hybrid system
recs_on_rated_hybrid = (
    ranked_hybrid
    .join(rated_universe, on=["user_id", "item_id"], how="inner")
)

eval_df_hybrid = (
    recs_on_rated_hybrid
    .withColumn("relevant", (col("rating") >= 4).cast("int"))
)

user_metrics_hybrid = (
    eval_df_hybrid
    .groupBy("user_id")
    .agg(
        (sum("relevant") / lit(top_k)).alias("precision"),
        sum("relevant").alias("hits"),
        count("*").alias("rated_recommended")
    )
    .join(
        rated_universe
        .filter(col("rating") >= 4)
        .groupBy("user_id")
        .count()
        .withColumnRenamed("count", "total_relevant"),
        on="user_id",
        how="left"
    )
    .fillna(0)
    .withColumn(
        "recall",
        when(col("total_relevant") > 0,
             col("hits") / col("total_relevant"))
        .otherwise(lit(0))
    )
)

avg_metrics_hybrid = user_metrics_hybrid.agg(
    avg("precision").alias(f"avg_precision@{top_k}"),
    avg("recall").alias(f"avg_recall@{top_k}")
)

print(f"\n===== HYBRID RESULTS (ALS weight={alpha}) =====")
avg_metrics_hybrid.show()


===== HYBRID RESULTS (ALS weight=1) =====


+-------------------+------------------+
| avg_precision@1000|   avg_recall@1000|
+-------------------+------------------+
|0.00483767228177639|0.6944090836057134|
+-------------------+------------------+

